In [ ]:
!pip install -q crewai

import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

from crewai import LLM
llm = LLM(
    model="gemini/gemini-flash-lite-latest",
    temperature=0.5
    )


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/

In [ ]:
article = """Why Every Student Should Learn AI Agents in 2026

AI agents are programs that can plan, use tools and complete multi-step tasks
on their own. Unlike a chatbot that only answers questions, an agent can search
the web, read files, write reports and even work with other agents as a team.

Companies are already using agents for customer support, resume screening and
bug triage, saving hundreds of hours every month. This is creating demand for
developers who understand how to design them.

The good news is that you don't need expensive tools to start. Frameworks like
CrewAI and free models like Gemini let students build real projects on Google
Colab. The key skills are clear prompt writing, breaking problems into steps,
and knowing when to let code make decisions instead of the AI.

Students who start now will have a strong advantage in placements, because
very few freshers can show working agent projects on their resume."""

In [ ]:
from crewai import Agent, Task, Crew

async def run_crew(role, goal, instructions, text):
  agent = Agent(
      role = role,
      goal = goal,
      backstory = "You are an experienced " + role + ".",
      llm = llm,
      verbose = False
  )
  task = Task(
      description = instructions + "\n\nContent:\n"+ text,
      expected_output = "The finished piece, ready to post.",
      agent = agent
  )

  crew = Crew(
      agents = [agent],
      tasks = [task],
      verbose = False
  )
  result = await crew.kickoff_async()
  return result.raw

In [ ]:
from crewai.flow.flow import Flow, start, listen, and_

class RepurposeFlow(Flow):

    @start()
    def load_article(self):
        print("📄 Article loaded:", len(self.state["article"]), "characters")

    @listen(load_article)
    def extract_points(self):
        points = llm.call(
            "List the 5 most important points from this article as short bullets.\n\n"
            + self.state["article"]
        )
        self.state["points"] = points.strip()
        print("Key points:\n" + self.state["points"])

    @listen(extract_points)
    async def write_linkedin(self):
        self.state["linkedin"] = await run_crew(
            role="LinkedIn Content Writer",
            goal="Write engaging LinkedIn posts",
            instructions="Write a LinkedIn post of about 150 words from these points. "
                         "Start with a strong one-line hook. End with a question.",
            text=self.state["points"],
        )

    @listen(extract_points)
    async def write_twitter(self):
        self.state["twitter"] = await run_crew(
            role="Twitter Content Writer",
            goal="Write punchy tweet threads",
            instructions="Write a thread of exactly 6 tweets from these points. "
                         "Number them 1/6 to 6/6. Each tweet under 280 characters.",
            text=self.state["points"],
        )

    @listen(extract_points)
    async def write_newsletter(self):
        self.state["newsletter"] = await run_crew(
            role="Newsletter Editor",
            goal="Write friendly newsletter sections",
            instructions="Write a newsletter section: a 2-line friendly intro, "
                         "then 3 bullet points, then a one-line sign-off.",
            text=self.state["points"],
        )

    @listen(and_(write_linkedin, write_twitter, write_newsletter))
    def publish_pack(self):
        for name in ["linkedin", "twitter", "newsletter"]:
            with open(name + ".md", "w") as f:
                f.write(self.state[name])
        print("Saved: linkedin.md, twitter.md, newsletter.md")

In [ ]:
flow = RepurposeFlow()
await flow.kickoff_async(inputs={"article": article})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: RepurposeFlow                                                                                            │
│  ID: f7e29481-f379-4b97-bdcb-7d84ccb7eaf4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: RepurposeFlow                                                                                            │
│  ID: f7e29481-f379-4b97-bdcb-7d84ccb7eaf4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_article                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📄 Article loaded: 930 characters


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: extract_points                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Saved: linkedin.md, twitter.md, newsletter.md


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: write_twitter                                                                                          │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_pack                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_pack                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: RepurposeFlow                                                                                            │
│  ID: f7e29481-f379-4b97-bdcb-7d84ccb7eaf4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from IPython.display import Markdown, display

for name in ["linkedin", "twitter", "newsletter"]:
    print("=" * 20, name.upper(), "=" * 20)
    with open(name + ".md") as f:
        display(Markdown(f.read()))

==================== LINKEDIN ====================


AI agents aren't just the future—they are the ultimate career cheat code for students right now. 

Unlike traditional software, autonomous AI agents can plan, use tools, and independently execute complex, multi-step tasks. Companies are already rushing to adopt them to automate bug triage and customer support, creating a massive skills gap.

The best part? You don't need a massive budget to get ahead. Using accessible tools like CrewAI and Google Colab, students can build production-ready projects completely for free. 

Mastering this comes down to three core skills: precise prompting, breaking down ambiguous problems, and knowing when to write code versus letting the AI decide. 

Add an agent project to your portfolio today, and watch how fast you stand out in the upcoming job market.

Which framework are you using to build your first AI agent?

==================== TWITTER ====================


1/6 AI agents are changing everything. Unlike traditional AI, they can plan, use tools, and complete complex multi-step tasks completely independently. A massive shift is underway. 🧵👇

2/6 Industry demand is exploding. Companies are rapidly adopting agents to automate heavy workflows like customer support and bug triage, saving countless hours. 

3/6 The best part? They are fully accessible. Students can now build real, working agent projects for free using powerful frameworks like CrewAI and Google Colab. 

4/6 To succeed, you need the right core skills: master clear prompting, learn to break down big problems, and balance hard code with autonomous AI decisions. 

5/6 The career advantage is huge. Students who start building agent projects right now will have a massive edge when entering the job market. 

6/6 The future belongs to those who build with AI, not just chat with it. What agent project are you going to start building today? Let's go! 🚀

==================== NEWSLETTER ====================


Hey everyone! Ready to dive into the future of tech? Here is a quick look at why AI agents are making waves right now.

* **Autonomous capability:** AI agents can plan, use tools, and complete multi-step tasks independently.
* **Industry demand:** Companies are using agents to save time on tasks like customer support and bug triage.
* **Accessible tools & skills:** Students can build free projects using frameworks like CrewAI and Google Colab, gaining a major career advantage along the way.

Happy building, and see you in the next issue!